# 01 — State Data Collection

Merges all state-level data sources into a single CSV for downstream notebooks:
- **HUD 2023 PIT counts**: total, sheltered, unsheltered, veteran homeless by state
- **Census ACS 5-year 2022**: income, poverty, Gini, rent burden, race/ethnicity, veteran %
- **SAMHSA NSDUH 2021-2022**: AMI prevalence, drug use disorder rate (embedded)
- **HUD FMR 2023**: median 1BR fair market rent (embedded)
- **NOAA climate normals**: average January temperature (embedded)
- **HUD CoC grants**: federal spending by state (embedded)

Output: `data/processed/merged_state_data.csv`


In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
from state_data import get_merged_state_data

OUT = ROOT / 'data' / 'processed'
OUT.mkdir(parents=True, exist_ok=True)


In [2]:
df = get_merged_state_data()
print(f'Merged state data: {len(df)} states, {len(df.columns)} columns')
print(f'Columns: {df.columns.tolist()}')


Census API unavailable (Expecting value: line 2 column 1 (char 1)), using embedded fallback.
Merged state data: 51 states, 26 columns
Columns: ['state_name', 'total_homeless', 'sheltered_homeless', 'unsheltered_homeless', 'veteran_homeless', 'population_millions', 'state_postal', 'state_fips', 'jan_temp_f', 'median_1br_fmr', 'ami_pct', 'drug_disorder_pct', 'overdose_rate_per_100k', 'hud_grants_millions', 'homeless_rate_per_10k', 'unsheltered_pct', 'spending_per_homeless', 'median_income', 'poverty_rate', 'gini_coefficient', 'rent_burden_pct', 'pct_white', 'pct_black', 'pct_hispanic', 'pct_asian', 'veteran_pct']


In [3]:
out_path = OUT / 'merged_state_data.csv'
df.to_csv(out_path, index=False)
print(f'Saved to {out_path}')

print(f'\nTop 5 states by homeless rate per 10k:')
print(df.nlargest(5, 'homeless_rate_per_10k')[['state_name', 'homeless_rate_per_10k', 'total_homeless']].to_string(index=False))
print(f'\nTotal US homeless count: {df["total_homeless"].sum():,}')


Saved to /Users/mckornfield/repo/homelessness-stats/data/processed/merged_state_data.csv

Top 5 states by homeless rate per 10k:
          state_name  homeless_rate_per_10k  total_homeless
District of Columbia                  94.51            6521
            New York                  52.44          103200
              Oregon                  47.50           20142
          California                  46.48          181399
              Hawaii                  45.96            6618

Total US homeless count: 669,675


In [4]:
preview = df[['state_name', 'state_postal', 'total_homeless', 'sheltered_homeless',
              'unsheltered_homeless', 'homeless_rate_per_10k', 'unsheltered_pct']]
preview = preview.sort_values('homeless_rate_per_10k', ascending=False)
preview


,state_name,state_postal,total_homeless,sheltered_homeless,unsheltered_homeless,homeless_rate_per_10k,unsheltered_pct
8,District of Columbia,DC,6521,5471,1050,94.51,16.1
32,New York,NY,103200,97700,5500,52.44,5.3
37,Oregon,OR,20142,9424,10718,47.50,53.2
4,California,CA,181399,113895,67504,46.48,37.2
11,Hawaii,HI,6618,3418,3200,45.96,48.4
47,Washington,WA,28036,14701,13335,36.22,47.6
21,Massachusetts,MA,23608,21808,1800,33.58,7.6
1,Alaska,AK,2369,1800,569,32.45,24.0
19,Maine,ME,4126,3326,800,29.90,19.4
5,Colorado,CO,14439,8892,5547,24.72,38.4
